In [3]:
pip install  tensorflow 


Note: you may need to restart the kernel to use updated packages.


In [4]:
train_dataset = tf.keras.utils.image_dataset_from_directory(
    "dataset",
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(128, 128),
    batch_size=32
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    "dataset",
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(128, 128),
    batch_size=32
)

Found 87028 files belonging to 2 classes.
Using 69623 files for training.
Found 87028 files belonging to 2 classes.
Using 17405 files for validation.


In [5]:
print(train_dataset.class_names)

['asl_alphabet_test', 'asl_alphabet_train']


In [6]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import os

## Normalize the Images

Neural networks train better when pixel values are between 0 and 1 instead of 0 and 255.

In [7]:
normalization_layer = tf.keras.layers.Rescaling(1./255)

train_dataset = train_dataset.map(
    lambda x, y: (normalization_layer(x), y)
)

validation_dataset = validation_dataset.map(
    lambda x, y: (normalization_layer(x), y)
)

## Optimize the Dataset

This helps TensorFlow load data more efficiently during training.


In [8]:
AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)
validation_dataset = validation_dataset.prefetch(buffer_size=AUTOTUNE)

In [9]:
for images, labels in train_dataset.take(1):
    print(images.shape)
    print(labels.shape)

(32, 128, 128, 3)
(32,)


In [10]:
class_names = train_dataset.class_names
print(class_names)

AttributeError: '_PrefetchDataset' object has no attribute 'class_names'

In [ ]:
normalization_layer = tf.keras.layers.Rescaling(1./255)

train_dataset = train_dataset.map(
    lambda x, y: (normalization_layer(x), y)
)

validation_dataset = validation_dataset.map(
    lambda x, y: (normalization_layer(x), y)
)

In [12]:
print(type(train_dataset))

<class 'tensorflow.python.data.ops.prefetch_op._PrefetchDataset'>


In [13]:
temp_dataset = tf.keras.utils.image_dataset_from_directory(
    "dataset",   # Change to "../dataset" if that's your actual path
    image_size=(128, 128),
    batch_size=32
)

print(type(temp_dataset))
print(hasattr(temp_dataset, "class_names"))


Found 87028 files belonging to 2 classes.
<class 'tensorflow.python.data.ops.prefetch_op._PrefetchDataset'>
True


In [14]:
import os

class_names = sorted([
    d for d in os.listdir("dataset")
    if os.path.isdir(os.path.join("dataset", d))
])

print(class_names)


['asl_alphabet_test', 'asl_alphabet_train']


In [15]:
import os

print(os.listdir("dataset/asl_alphabet_train"))

['asl_alphabet_train']


In [20]:
train_dataset = tf.keras.utils.image_dataset_from_directory(
    "dataset/asl_alphabet_train/asl_alphabet_train",
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(128, 128),
    batch_size=32
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    "dataset/asl_alphabet_train/asl_alphabet_train",
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(128, 128),
    batch_size=32
)

Found 87000 files belonging to 29 classes.
Using 69600 files for training.
Found 87000 files belonging to 29 classes.
Using 17400 files for validation.


In [21]:
class_names = train_dataset.class_names
print(class_names)

['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'del', 'nothing', 'space']


In [23]:
import os

print(os.listdir("dataset/asl_alphabet_train/asl_alphabet_train")[:10])

['A', 'B', 'C', 'D', 'del', 'E', 'F', 'G', 'H', 'I']


### BUILDING AN CNN MODEL 

In [24]:
from tensorflow.keras import models, layers 


model = models.Sequential([
    
    # Input Layer
    layers.Input(shape=(128, 128, 3)),
    
    # First Convolution Block
    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    
    # Second Convolution Block
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    
    # Third Convolution Block
    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    
    # Convert Feature Maps to 1D
    layers.Flatten(),
    
    # Fully Connected Layer
    layers.Dense(256, activation='relu'),
    
    # Prevent Overfitting
    layers.Dropout(0.5),
    
    # Output Layer
    layers.Dense(len(class_names), activation='softmax')
])

In [26]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 126, 126, 32)        │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 63, 63, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 61, 61, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 30, 30, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 28, 28, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 14, 14, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 25088)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 256)                 │       6,422,784 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 29)                  │           7,453 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 6,523,485 (24.89 MB)

 Trainable params: 6,523,485 (24.89 MB)

 Non-trainable params: 0 (0.00 B)

In [27]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=20
)

Epoch 1/20
 990/2175 ━━━━━━━━━━━━━━━━━━━━ 2:11 111ms/step - accuracy: 0.2269 - loss: 6.6335 

In [ ]:
model.save("silent_voice_model.keras")

In [ ]:
import matplotlib.pyplot as plt

# Accuracy
plt.figure(figsize=(8,5))
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.title("Model Accuracy")
plt.show()

# Loss
plt.figure(figsize=(8,5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("Model Loss")
plt.show()